# 房价预测竞赛实现

## 理解数据

In [1]:
import pandas as pd
train_data = pd.read_csv("data/house-prices-advanced-regression-techniques/train.csv")
train_data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [2]:
train_data.shape,train_data.columns[-5:],train_data.columns.tolist()

((1460, 81),
 Index(['MoSold', 'YrSold', 'SaleType', 'SaleCondition', 'SalePrice'], dtype='object'),
 ['Id',
  'MSSubClass',
  'MSZoning',
  'LotFrontage',
  'LotArea',
  'Street',
  'Alley',
  'LotShape',
  'LandContour',
  'Utilities',
  'LotConfig',
  'LandSlope',
  'Neighborhood',
  'Condition1',
  'Condition2',
  'BldgType',
  'HouseStyle',
  'OverallQual',
  'OverallCond',
  'YearBuilt',
  'YearRemodAdd',
  'RoofStyle',
  'RoofMatl',
  'Exterior1st',
  'Exterior2nd',
  'MasVnrType',
  'MasVnrArea',
  'ExterQual',
  'ExterCond',
  'Foundation',
  'BsmtQual',
  'BsmtCond',
  'BsmtExposure',
  'BsmtFinType1',
  'BsmtFinSF1',
  'BsmtFinType2',
  'BsmtFinSF2',
  'BsmtUnfSF',
  'TotalBsmtSF',
  'Heating',
  'HeatingQC',
  'CentralAir',
  'Electrical',
  '1stFlrSF',
  '2ndFlrSF',
  'LowQualFinSF',
  'GrLivArea',
  'BsmtFullBath',
  'BsmtHalfBath',
  'FullBath',
  'HalfBath',
  'BedroomAbvGr',
  'KitchenAbvGr',
  'KitchenQual',
  'TotRmsAbvGrd',
  'Functional',
  'Fireplaces',
  'Firepla

In [3]:
test_data = pd.read_csv("data/house-prices-advanced-regression-techniques/test.csv")
print(F"测试集形状{test_data.shape}")
test_data.head(),test_data.columns[-5:]

测试集形状(1459, 80)


(     Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
 0  1461          20       RH         80.0    11622   Pave   NaN      Reg   
 1  1462          20       RL         81.0    14267   Pave   NaN      IR1   
 2  1463          60       RL         74.0    13830   Pave   NaN      IR1   
 3  1464          60       RL         78.0     9978   Pave   NaN      IR1   
 4  1465         120       RL         43.0     5005   Pave   NaN      IR1   
 
   LandContour Utilities  ... ScreenPorch PoolArea PoolQC  Fence MiscFeature  \
 0         Lvl    AllPub  ...         120        0    NaN  MnPrv         NaN   
 1         Lvl    AllPub  ...           0        0    NaN    NaN        Gar2   
 2         Lvl    AllPub  ...           0        0    NaN  MnPrv         NaN   
 3         Lvl    AllPub  ...           0        0    NaN    NaN         NaN   
 4         HLS    AllPub  ...         144        0    NaN    NaN         NaN   
 
   MiscVal MoSold  YrSold  SaleType  SaleCondition  
 

In [4]:
sample_submission=pd.read_csv("data/house-prices-advanced-regression-techniques/sample_submission.csv")
print(F"提交文件的形状{sample_submission.shape}")
sample_submission.columns,sample_submission.head(),(test_data["Id"] == sample_submission["Id"]).all()

提交文件的形状(1459, 2)


(Index(['Id', 'SalePrice'], dtype='object'),
      Id      SalePrice
 0  1461  169277.052498
 1  1462  187758.393989
 2  1463  183583.683570
 3  1464  179317.477511
 4  1465  150730.079977,
 np.True_)

## 划分特征和标签

In [5]:
train_features=train_data.drop(columns=["Id","SalePrice"])
train_labels=train_data["SalePrice"]
test_features=test_data.drop(columns=["Id"])
f'训练特征:{train_features.shape}',f'训练标签:{train_labels.shape}',f'测试特征:{test_features.shape}'

('训练特征:(1460, 79)', '训练标签:(1460,)', '测试特征:(1459, 79)')

In [6]:
train_data.dtypes.value_counts()

object     43
int64      35
float64     3
Name: count, dtype: int64

## 缺失值处理

In [7]:
missing_count = train_features.isnull().sum()
missing_count=missing_count[missing_count>0]
missing_count=missing_count.sort_values(ascending=False)

missing_count,len(missing_count)

(PoolQC          1453
 MiscFeature     1406
 Alley           1369
 Fence           1179
 MasVnrType       872
 FireplaceQu      690
 LotFrontage      259
 GarageType        81
 GarageYrBlt       81
 GarageFinish      81
 GarageQual        81
 GarageCond        81
 BsmtExposure      38
 BsmtFinType2      38
 BsmtQual          37
 BsmtCond          37
 BsmtFinType1      37
 MasVnrArea         8
 Electrical         1
 dtype: int64,
 19)

## 合并训练特征和测试特征

In [8]:
train_features.columns.equals(test_features.columns)

True

In [9]:
n_train=train_features.shape[0]
all_features=pd.concat([train_features,test_features],axis=0,ignore_index=True)

n_train,all_features.shape,all_features.iloc[:n_train].shape,all_features.iloc[n_train:].shape

(1460, (2919, 79), (1460, 79), (1459, 79))

## 数值特征的尺度问题

In [10]:
numeric_features=all_features.select_dtypes(include=["number"]).columns
category_featrures=all_features.select_dtypes(exclude=["number"]).columns
len(numeric_features),len(category_featrures)

(36, 43)

In [11]:
all_features[["OverallQual", "LotArea", "YearBuilt"]].describe()

,OverallQual,LotArea,YearBuilt
count,2919.000000,2919.000000,2919.000000
mean,6.089072,10168.114080,1971.312778
std,1.409947,7886.996359,30.291442
min,1.000000,1300.000000,1872.000000
25%,5.000000,7478.000000,1953.500000
50%,6.000000,9453.000000,1973.000000
75%,7.000000,11570.000000,2001.000000
max,10.000000,215245.000000,2010.000000


## 数值标准化

In [12]:
all_features[numeric_features].isnull().sum().sum()

np.int64(678)

In [13]:
all_features[numeric_features]=all_features[numeric_features].apply(lambda x:(x-x.mean())/x.std())
all_features["LotArea"].describe()

count    2.919000e+03
mean     2.921039e-17
std      1.000000e+00
min     -1.124397e+00
25%     -3.410822e-01
50%     -9.067002e-02
75%      1.777465e-01
max      2.600190e+01
Name: LotArea, dtype: float64

In [14]:
all_features[numeric_features]=all_features[numeric_features].fillna(0)
all_features[numeric_features].isnull().sum().sum()

np.int64(0)

## 独热编码

In [15]:
all_features["MSZoning"].head()

0    RL
1    RL
2    RL
3    RL
4    RL
Name: MSZoning, dtype: object

In [16]:
pd.get_dummies(
    all_features["MSZoning"],
    prefix="MSZoning",
    dummy_na=True,
    dtype=float
).head()

,MSZoning_C (all),MSZoning_FV,MSZoning_RH,MSZoning_RL,MSZoning_RM,MSZoning_nan
0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0


In [17]:
print(f"编码前形状:{all_features.shape}")

all_features=pd.get_dummies(
    all_features,
    dummy_na=True,
    dtype=float
)
print(f"编码后形状{all_features.shape}")
print(f"剩余缺失值：{all_features.isnull().sum().sum()}")
print(all_features.dtypes.value_counts())



编码前形状:(2919, 79)
编码后形状(2919, 330)
剩余缺失值：0
float64    330
Name: count, dtype: int64


## 拆分数据集，并将其转换成张量

In [18]:
processed_train_data=all_features.iloc[:n_train]
processed_test_data=all_features.iloc[n_train:]

print(processed_train_data.shape)
print(processed_test_data.shape)

(1460, 330)
(1459, 330)


In [19]:
import torch

train_features=torch.tensor(processed_train_data.values,dtype=torch.float32)
test_features=torch.tensor(processed_test_data.values,dtype=torch.float32)
train_labels=torch.tensor(train_labels.values,dtype=torch.float32).reshape(-1,1)
train_features.shape,test_features.shape,train_labels.shape

(torch.Size([1460, 330]), torch.Size([1459, 330]), torch.Size([1460, 1]))

## 建立模型

In [20]:
from torch import nn
num_inputs=train_features.shape[1]

net=nn.Sequential(nn.Linear(num_inputs,1))
train_features.shape[0],train_features.shape[1]

(1460, 330)

In [21]:
predictions = net(train_features[:5])
print("输入形状：", train_features[:5].shape)
print("输出形状：", predictions.shape)
print("预测结果：", predictions)

输入形状： torch.Size([5, 330])
输出形状： torch.Size([5, 1])
预测结果： tensor([[-0.1178],
        [-0.3099],
        [-0.1099],
        [ 0.2565],
        [-0.0167]], grad_fn=<AddmmBackward0>)


In [22]:
loss=nn.MSELoss()
predictions=net(train_features[5:])
loss=loss(predictions,train_labels[5:])

print("预测值：", predictions)
print("真实值：", train_labels[:5])
print("均方误差：", loss.item())

预测值： tensor([[-0.3044],
        [-0.1949],
        [-0.3124],
        ...,
        [-0.3289],
        [ 0.0291],
        [-0.0348]], grad_fn=<AddmmBackward0>)
真实值： tensor([[208500.],
        [181500.],
        [223500.],
        [140000.],
        [250000.]])
均方误差： 39030181888.0


In [23]:

def log_rmse(net,features,labels):
    loss=nn.MSELoss()
    with torch.no_grad():
        predictions=net(features)

        predictions=torch.clamp(predictions,min=1)# 防止预测值小于等于0，因为 log(0) 和负数的 log 无意义
        predictions=torch.log(predictions)
        labels=torch.log(labels)

        rmse=torch.sqrt(loss(predictions,labels))

    return rmse.item()

log_rmse(net, train_features, train_labels)

12.030646324157715

## 划分数据集

In [24]:
torch.manual_seed(42)

num_samples = train_features.shape[0]
num_valid = int(num_samples * 0.2)

indices = torch.randperm(num_samples)

valid_indices = indices[:num_valid]
train_indices = indices[num_valid:]

In [25]:
X_train=train_features[train_indices]
y_train=train_labels[train_indices]

X_valid=train_features[valid_indices]
y_valid=train_labels[valid_indices]

print(X_train.shape, y_train.shape)
print(X_valid.shape, y_valid.shape)

torch.Size([1168, 330]) torch.Size([1168, 1])
torch.Size([292, 330]) torch.Size([292, 1])


## 创建小批量数据

In [26]:
from torch.utils.data import TensorDataset,DataLoader

train_dataset=TensorDataset(X_train,y_train)
batch_size=64
train_loader=DataLoader(
    train_dataset,
    shuffle=True,
    batch_size=batch_size
)

In [27]:
batch_X, batch_y = next(iter(train_loader))

print(batch_X.shape)
print(batch_y.shape)
print("每轮批次数：", len(train_loader))

torch.Size([64, 330])
torch.Size([64, 1])
每轮批次数： 19


## 参数更新

In [28]:
torch.manual_seed(42)

net = nn.Sequential(
    nn.Linear(num_inputs, 1)
)

loss = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr=5)

In [29]:
loss_before = loss(net(batch_X), batch_y)

optimizer.zero_grad()
predictions=net(batch_X)
batch_loss = loss(predictions,batch_y)

batch_loss.backward()
optimizer.step()

In [30]:
loss_after = loss(net(batch_X), batch_y)

print("更新前：", loss_before.item())
print("更新后：", loss_after.item())

更新前： 28996354048.0
更新后： 28921516032.0


## 完整流程

In [31]:
torch.manual_seed(42)

net = nn.Sequential(
    nn.Linear(num_inputs, 1)
)

optimizer = torch.optim.Adam(
    net.parameters(),
    lr=5
)

num_epochs = 100

In [32]:
for epoch in range(num_epochs):
    net.train()

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()

        predictions = net(batch_X)
        batch_loss = loss(predictions,batch_y)

        batch_loss.backward()
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        train_score = log_rmse(net, X_train, y_train)
        valid_score = log_rmse(net, X_valid, y_valid)

        print(
            f"epoch {epoch + 1:3d}, "
            f"train log RMSE {train_score:.4f}, "
            f"valid log RMSE {valid_score:.4f}"
        )

epoch  10, train log RMSE 1.4927, valid log RMSE 1.5125
epoch  20, train log RMSE 0.8783, valid log RMSE 0.8956
epoch  30, train log RMSE 0.5610, valid log RMSE 0.5750
epoch  40, train log RMSE 0.3741, valid log RMSE 0.3838
epoch  50, train log RMSE 0.2654, valid log RMSE 0.2698
epoch  60, train log RMSE 0.2076, valid log RMSE 0.2063
epoch  70, train log RMSE 0.1816, valid log RMSE 0.1755
epoch  80, train log RMSE 0.1715, valid log RMSE 0.1621
epoch  90, train log RMSE 0.1683, valid log RMSE 0.1569
epoch 100, train log RMSE 0.1671, valid log RMSE 0.1547


## 交叉验证

In [33]:
k = 5

generator = torch.Generator().manual_seed(42)
shuffled_indices = torch.randperm(
    train_features.shape[0],
    generator=generator
)

fold_indices = torch.chunk(shuffled_indices, k)

for i, fold in enumerate(fold_indices):
    print(f"第{i + 1}折：{len(fold)}条")

第1折：292条
第2折：292条
第3折：292条
第4折：292条
第5折：292条


In [34]:
valid_indices = fold_indices[0]
train_indices = torch.cat(fold_indices[1:])

fold_X_train = train_features[train_indices]
fold_y_train = train_labels[train_indices]

fold_X_valid = train_features[valid_indices]
fold_y_valid = train_labels[valid_indices]

print(fold_X_train.shape, fold_y_train.shape)
print(fold_X_valid.shape, fold_y_valid.shape)

torch.Size([1168, 330]) torch.Size([1168, 1])
torch.Size([292, 330]) torch.Size([292, 1])


In [35]:
def get_net():
    torch.manual_seed(42)

    model = nn.Sequential(
        nn.Linear(num_inputs, 1)
    )

    return model

In [36]:
net_1 = get_net()
net_2 = get_net()

print(net_1 is net_2)
print(torch.equal(net_1[0].weight, net_2[0].weight))

False
True


In [37]:
def train_one_fold(
    X_train,
    y_train,
    X_valid,
    y_valid,
    num_epochs=100,
    learning_rate=5,
    batch_size=64
):
    model = get_net()

    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    for epoch in range(num_epochs):
        model.train()

        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()

            predictions = model(batch_X)
            batch_loss = loss(predictions, batch_y)

            batch_loss.backward()
            optimizer.step()

    train_score = log_rmse(model, X_train, y_train)
    valid_score = log_rmse(model, X_valid, y_valid)

    return model, train_score, valid_score

In [38]:
fold_model, train_score, valid_score = train_one_fold(
    fold_X_train,
    fold_y_train,
    fold_X_valid,
    fold_y_valid
)

print("第一折训练分数：", train_score)
print("第一折验证分数：", valid_score)

第一折训练分数： 0.16711127758026123
第一折验证分数： 0.15471318364143372


In [39]:
train_scores = []
valid_scores = []

for i in range(k):
    valid_indices = fold_indices[i]

    train_indices = torch.cat([
        fold_indices[j]
        for j in range(k)
        if j != i
    ])

    fold_X_train = train_features[train_indices]
    fold_y_train = train_labels[train_indices]

    fold_X_valid = train_features[valid_indices]
    fold_y_valid = train_labels[valid_indices]

    _, train_score, valid_score = train_one_fold(
        fold_X_train,
        fold_y_train,
        fold_X_valid,
        fold_y_valid
    )

    train_scores.append(train_score)
    valid_scores.append(valid_score)

    print(
        f"第{i + 1}折："
        f"train={train_score:.4f}, "
        f"valid={valid_score:.4f}"
    )

第1折：train=0.1671, valid=0.1547
第2折：train=0.1606, valid=0.1988
第3折：train=0.1630, valid=0.2340
第4折：train=0.1665, valid=0.1602
第5折：train=0.1712, valid=0.1395


In [40]:
mean_train_score = sum(train_scores) / len(train_scores)
mean_valid_score = sum(valid_scores) / len(valid_scores)

print(f"平均训练 log RMSE：{mean_train_score:.4f}")
print(f"平均验证 log RMSE：{mean_valid_score:.4f}")

平均训练 log RMSE：0.1657
平均验证 log RMSE：0.1774


In [41]:
print("各折验证分数：", valid_scores)
print("最好的一折：", min(valid_scores))
print("最差的一折：", max(valid_scores))

各折验证分数： [0.15471318364143372, 0.19879105687141418, 0.23398706316947937, 0.16019251942634583, 0.1394796222448349]
最好的一折： 0.1394796222448349
最差的一折： 0.23398706316947937


## 用全部训练数据训练最终模型

In [42]:
full_train_dataset = TensorDataset(
    train_features,
    train_labels
)

full_train_loader = DataLoader(
    full_train_dataset,
    batch_size=64,
    shuffle=True
)

In [43]:
final_model = get_net()

final_optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=5
)

num_epochs = 100

for epoch in range(num_epochs):
    final_model.train()

    for batch_X, batch_y in full_train_loader:
        final_optimizer.zero_grad()

        predictions = final_model(batch_X)
        batch_loss = loss(predictions, batch_y)

        batch_loss.backward()
        final_optimizer.step()

    if (epoch + 1) % 20 == 0:
        train_score = log_rmse(
            final_model,
            train_features,
            train_labels
        )

        print(
            f"epoch {epoch + 1:3d}, "
            f"full train log RMSE={train_score:.4f}"
        )

epoch  20, full train log RMSE=0.7252
epoch  40, full train log RMSE=0.2777
epoch  60, full train log RMSE=0.1756
epoch  80, full train log RMSE=0.1657
epoch 100, full train log RMSE=0.1623


In [44]:
final_train_score = log_rmse(
    final_model,
    train_features,
    train_labels
)

print("最终训练分数：", final_train_score)

最终训练分数： 0.1623125523328781


## 预测测试集

In [45]:
final_model.eval()

with torch.no_grad():
    test_predictions = final_model(test_features)

    # 房价不能小于等于0
    test_predictions = torch.clamp(
        test_predictions,
        min=1
    )

In [46]:
print("预测形状：", test_predictions.shape)
print("最低预测：", test_predictions.min().item())
print("最高预测：", test_predictions.max().item())
print("平均预测：", test_predictions.mean().item())
print("前5个预测：", test_predictions[:5])

预测形状： torch.Size([1459, 1])
最低预测： 24215.98046875
最高预测： 578876.9375
平均预测： 177173.75
前5个预测： tensor([[119376.7734],
        [154008.6562],
        [198558.5000],
        [217027.0156],
        [177362.2344]])


In [47]:
submission = sample_submission.copy()

submission["SalePrice"] = (
    test_predictions
    .squeeze(1)
    .cpu()
    .numpy()
)

In [48]:
print(submission.shape)
print(submission.head())
print(submission.isnull().sum())
print((submission["SalePrice"] > 0).all())
print((submission["Id"] == test_data["Id"]).all())

(1459, 2)
     Id      SalePrice
0  1461  119376.773438
1  1462  154008.656250
2  1463  198558.500000
3  1464  217027.015625
4  1465  177362.234375
Id           0
SalePrice    0
dtype: int64
True
True


In [49]:
submission_path = (
    "data/house-prices-advanced-regression-techniques/"
    "submission.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print("已保存：", submission_path)

已保存： data/house-prices-advanced-regression-techniques/submission.csv


In [50]:
saved_submission = pd.read_csv(submission_path)

print(saved_submission.shape)
print(saved_submission.head())
print(saved_submission.columns.tolist())

(1459, 2)
     Id  SalePrice
0  1461  119376.77
1  1462  154008.66
2  1463  198558.50
3  1464  217027.02
4  1465  177362.23
['Id', 'SalePrice']


# 优化一：预测对数房价

## 转换标签

In [51]:
log_train_labels=torch.log(train_labels)

print(f"原始标签形状{train_labels.shape}\n,对数标签形状{log_train_labels.shape}")

print("原始房价范围",train_labels.min().item(),train_labels.max().item())

print("对数房价范围",log_train_labels.min().item(),log_train_labels.max().item())

print('前五个房价',train_labels[:5])
print('前五个对数房价',log_train_labels[:5])

原始标签形状torch.Size([1460, 1])
,对数标签形状torch.Size([1460, 1])
原始房价范围 34900.0 755000.0
对数房价范围 10.46024227142334 13.534473419189453
前五个房价 tensor([[208500.],
        [181500.],
        [223500.],
        [140000.],
        [250000.]])
前五个对数房价 tensor([[12.2477],
        [12.1090],
        [12.3172],
        [11.8494],
        [12.4292]])


## 定义评价函数

In [52]:
loss = nn.MSELoss()
def log_target_rmse(model,features,log_labels):
    model.eval()

    with torch.no_grad():
        predicted_log_prices=model(features)

        rmse=torch.sqrt(loss(predicted_log_prices,log_labels))

    return rmse.item()


### 使用一折实验

In [53]:
valid_indices=fold_indices[0]
train_indices=torch.cat(fold_indices[1:])

log_X_train=train_features[train_indices]
log_y_train=log_train_labels[train_indices]

log_X_valid=train_features[valid_indices]
log_y_valid=log_train_labels[valid_indices]

print(log_X_train.shape, log_y_train.shape)
print(log_X_valid.shape, log_y_valid.shape)

torch.Size([1168, 330]) torch.Size([1168, 1])
torch.Size([292, 330]) torch.Size([292, 1])


In [54]:
def train_log_one_fold(
        X_train,
        log_y_train,
        X_valid,
        log_y_valid,
        num_epochs=100,
        learning_rate=0.01,
        batch_size=64
):
    model =get_net()
    train_dataset=TensorDataset(X_train,log_y_train)
    train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)

    optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

    for epoch in range(num_epochs):
        model.train()
        for batch_X,batch_log_y in train_loader:
            optimizer.zero_grad()
            predicted_log_y=model(batch_X)
            batch_loss=loss(predicted_log_y,batch_log_y)
            batch_loss.backward()
            optimizer.step()
    train_score = log_target_rmse(
        model,
        X_train,
        log_y_train
    )

    valid_score = log_target_rmse(
        model,
        X_valid,
        log_y_valid
    )

    return model, train_score, valid_score


In [55]:
log_model, log_train_score, log_valid_score = train_log_one_fold(
    log_X_train,
    log_y_train,
    log_X_valid,
    log_y_valid
)

print("对数模型第一折训练分数：", log_train_score)
print("对数模型第一折验证分数：", log_valid_score)

对数模型第一折训练分数： 0.09300830215215683
对数模型第一折验证分数： 0.14551548659801483


In [58]:
log_train_scores = []
log_valid_scores = []

for i in range(k):
    valid_indices = fold_indices[i]

    train_indices = torch.cat([
        fold_indices[j]
        for j in range(k)
        if j != i
    ])

    fold_X_train = train_features[train_indices]
    fold_log_y_train = log_train_labels[train_indices]

    fold_X_valid = train_features[valid_indices]
    fold_log_y_valid = log_train_labels[valid_indices]

    _, log_train_score, log_valid_score = train_log_one_fold(
        fold_X_train,
        fold_log_y_train,
        fold_X_valid,
        fold_log_y_valid
    )

    log_train_scores.append(log_train_score)
    log_valid_scores.append(log_valid_score)

    print(
        f"第{i + 1}折："
        f"train={log_train_score:.4f}, "
        f"valid={log_valid_score:.4f}"
    )

第1折：train=0.0930, valid=0.1455
第2折：train=0.0922, valid=0.2045
第3折：train=0.0900, valid=0.1690
第4折：train=0.0999, valid=0.1201
第5折：train=0.0976, valid=0.1228


In [59]:
mean_log_train_score = (
    sum(log_train_scores) / len(log_train_scores)
)

mean_log_valid_score = (
    sum(log_valid_scores) / len(log_valid_scores)
)

print(f"对数模型平均训练分数：{mean_log_train_score:.4f}")
print(f"对数模型平均验证分数：{mean_log_valid_score:.4f}")

对数模型平均训练分数：0.0945
对数模型平均验证分数：0.1524


## 完整代码

In [61]:
full_log_dataset=TensorDataset(train_features,log_train_labels)

full_train_loader=DataLoader(full_log_dataset,batch_size=64,shuffle=True)

final_model=get_net()

optimizer=torch.optim.Adam(final_model.parameters(),lr=0.01)

num_epochs=100

for epoch in range(num_epochs):
    final_model.train()

    for batch_X,batch_log_y in full_train_loader:
        optimizer.zero_grad()
        predicted_log_y=final_model(batch_X)
        batch_loss=loss(predicted_log_y,batch_log_y)
        batch_loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 0:
        score = log_target_rmse(
            final_model,
            train_features,
            log_train_labels
        )

        print(
            f"epoch {epoch + 1:3d}, "
            f"log RMSE={score:.4f}"
        )

epoch  20, log RMSE=0.1181
epoch  40, log RMSE=0.1022
epoch  60, log RMSE=0.0990
epoch  80, log RMSE=0.0986
epoch 100, log RMSE=0.0996


In [62]:
final_model.eval()

with torch.no_grad():
    predicted_log_prices = final_model(test_features)
    predicted_prices = torch.exp(predicted_log_prices)

In [63]:
print(predicted_log_prices.shape)
print("最低房价：", predicted_prices.min().item())
print("最高房价：", predicted_prices.max().item())
print("平均房价：", predicted_prices.mean().item())
print("前5个预测：", predicted_prices[:5])

torch.Size([1459, 1])
最低房价： 35256.68359375
最高房价： 1714270.0
平均房价： 182092.859375
前5个预测： tensor([[118906.0000],
        [156284.3906],
        [182391.4844],
        [205571.2969],
        [191859.8125]])


In [64]:
log_submission = sample_submission.copy()

log_submission["SalePrice"] = (
    predicted_prices
    .squeeze(1)
    .cpu()
    .numpy()
)

log_submission_path = (
    "data/house-prices-advanced-regression-techniques/"
    "submission_log_target.csv"
)

log_submission.to_csv(
    log_submission_path,
    index=False
)

In [65]:
print(log_submission.shape)
print(log_submission.head())
print(log_submission.isnull().sum())
print((log_submission["SalePrice"] > 0).all())
print((log_submission["Id"] == test_data["Id"]).all())

(1459, 2)
     Id      SalePrice
0  1461  118906.000000
1  1462  156284.390625
2  1463  182391.484375
3  1464  205571.296875
4  1465  191859.812500
Id           0
SalePrice    0
dtype: int64
True
True


In [ ]:
def train_log_one_fold(
    X_train,
    log_y_train,
    X_valid,
    log_y_valid,
    num_epochs=100,
    learning_rate=0.01,
    batch_size=64,
    weight_decay=0
):
    model =get_net()
    train_dataset=TensorDataset(X_train,log_y_train)
    train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)

    optimizer=torch.optim.Adam(
        [
            {
                'params':[model[0].weight],
                "weight_decay":weight_decay
            },
            {
                'params':[model[0].bias],
                'weight_decay':weight_decay
            }
        ],
        lr=learning_rate)

    for epoch in range(num_epochs):
        model.train()
        for batch_X,batch_log_y in train_loader:
            optimizer.zero_grad()
            predicted_log_y=model(batch_X)
            batch_loss=loss(predicted_log_y,batch_log_y)
            batch_loss.backward()
            optimizer.step()
    train_score = log_target_rmse(
        model,
        X_train,
        log_y_train
    )

    valid_score = log_target_rmse(
        model,
        X_valid,
        log_y_valid
    )

    return model, train_score, valid_score

In [71]:
weight_decay_values = [
    0,
    1e-4,
    1e-3,
    1e-2
]

for wd in weight_decay_values:
    _, train_score, valid_score = train_log_one_fold(
        log_X_train,
        log_y_train,
        log_X_valid,
        log_y_valid,
        weight_decay=wd
    )

    print(
        f"weight_decay={wd:g}, "
        f"train={train_score:.4f}, "
        f"valid={valid_score:.4f}, "
        f"gap={valid_score - train_score:.4f}"
    )

weight_decay=0, train=0.0930, valid=0.1455, gap=0.0525
weight_decay=0.0001, train=0.0934, valid=0.1436, gap=0.0502
weight_decay=0.001, train=0.0987, valid=0.1353, gap=0.0366
weight_decay=0.01, train=0.1357, valid=0.1481, gap=0.0124


In [72]:
def cross_validate_weight_decay(weight_decay):
    fold_train_scores = []
    fold_valid_scores = []

    for i in range(k):
        valid_indices = fold_indices[i]

        train_indices = torch.cat([
            fold_indices[j]
            for j in range(k)
            if j != i
        ])

        fold_X_train = train_features[train_indices]
        fold_log_y_train = log_train_labels[train_indices]

        fold_X_valid = train_features[valid_indices]
        fold_log_y_valid = log_train_labels[valid_indices]

        _, train_score, valid_score = train_log_one_fold(
            fold_X_train,
            fold_log_y_train,
            fold_X_valid,
            fold_log_y_valid,
            weight_decay=weight_decay
        )

        fold_train_scores.append(train_score)
        fold_valid_scores.append(valid_score)

    mean_train = (
        sum(fold_train_scores) / len(fold_train_scores)
    )
    mean_valid = (
        sum(fold_valid_scores) / len(fold_valid_scores)
    )

    return mean_train, mean_valid

In [73]:
weight_decay_results = []

for wd in weight_decay_values:
    mean_train, mean_valid = cross_validate_weight_decay(wd)

    weight_decay_results.append({
        "weight_decay": wd,
        "train_score": mean_train,
        "valid_score": mean_valid
    })

    print(
        f"weight_decay={wd:g}, "
        f"train={mean_train:.4f}, "
        f"valid={mean_valid:.4f}, "
        f"gap={mean_valid - mean_train:.4f}"
    )

weight_decay=0, train=0.0945, valid=0.1524, gap=0.0578
weight_decay=0.0001, train=0.0949, valid=0.1518, gap=0.0568
weight_decay=0.001, train=0.1002, valid=0.1510, gap=0.0508
weight_decay=0.01, train=0.1373, valid=0.1809, gap=0.0436


In [74]:
best_result = min(
    weight_decay_results,
    key=lambda result: result["valid_score"]
)

print("最佳结果：", best_result)

最佳结果： {'weight_decay': 0.001, 'train_score': 0.10022790580987931, 'valid_score': 0.15100837200880052}


In [75]:
full_log_dataset = TensorDataset(
    train_features,
    log_train_labels
)

full_log_loader = DataLoader(
    full_log_dataset,
    batch_size=64,
    shuffle=True
)
print(
    torch.equal(
        full_log_loader.dataset.tensors[1],
        log_train_labels
    )
)

True


In [76]:
final_log_wd_model = get_net()

final_log_wd_optimizer = torch.optim.Adam(
    [
        {
            "params": [final_log_wd_model[0].weight],
            "weight_decay": 1e-3
        },
        {
            "params": [final_log_wd_model[0].bias],
            "weight_decay": 0
        }
    ],
    lr=0.01
)

In [77]:
num_epochs = 100

for epoch in range(num_epochs):
    final_log_wd_model.train()

    for batch_X, batch_log_y in full_log_loader:
        final_log_wd_optimizer.zero_grad()

        predicted_log_y = final_log_wd_model(batch_X)

        batch_loss = loss(
            predicted_log_y,
            batch_log_y
        )

        batch_loss.backward()
        final_log_wd_optimizer.step()

    if (epoch + 1) % 20 == 0:
        train_score = log_target_rmse(
            final_log_wd_model,
            train_features,
            log_train_labels
        )

        print(
            f"epoch {epoch + 1:3d}, "
            f"train log RMSE={train_score:.4f}"
        )

epoch  20, train log RMSE=0.1197
epoch  40, train log RMSE=0.1054
epoch  60, train log RMSE=0.1034
epoch  80, train log RMSE=0.1034
epoch 100, train log RMSE=0.1069


In [78]:
final_log_wd_model.eval()

with torch.no_grad():
    predicted_test_log_prices = final_log_wd_model(
        test_features
    )

    predicted_test_prices = torch.exp(
        predicted_test_log_prices
    )

In [79]:
print("形状：", predicted_test_prices.shape)
print("最低：", predicted_test_prices.min().item())
print("最高：", predicted_test_prices.max().item())
print("平均：", predicted_test_prices.mean().item())

形状： torch.Size([1459, 1])
最低： 29414.99609375
最高： 1622083.5
平均： 182522.328125


In [80]:
wd_submission = sample_submission.copy()

wd_submission["SalePrice"] = (
    predicted_test_prices
    .squeeze(1)
    .cpu()
    .numpy()
)

wd_submission_path = (
    "data/house-prices-advanced-regression-techniques/"
    "submission_log_wd_1e-3.csv"
)

wd_submission.to_csv(
    wd_submission_path,
    index=False
)

In [81]:
print(wd_submission.shape)
print(wd_submission.head())
print(wd_submission.isnull().sum())
print((wd_submission["SalePrice"] > 0).all())
print((wd_submission["Id"] == test_data["Id"]).all())

(1459, 2)
     Id      SalePrice
0  1461  115077.414062
1  1462  146722.937500
2  1463  178193.406250
3  1464  201164.031250
4  1465  193136.250000
Id           0
SalePrice    0
dtype: int64
True
True
